# Chapter 3 — Exercise Solutions## MDPs and Markov Chains[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 3.1: Extended Clickstream MDP with Wishlist

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# States: Homepage, Product, Wishlist, Cart, Purchase, Exitstates = ['Homepage', 'Product', 'Wishlist', 'Cart', 'Purchase', 'Exit']S = len(states)sidx = {s: i for i, s in enumerate(states)}# Transition matrix P[state, action, next_state]# Actions: 0=no_rec, 1=aggressive_recP = np.zeros((S, 2, S))# Homepage transitionsP[0, 0] = [0.25, 0.40, 0.05, 0.10, 0.00, 0.20]  # no_recP[0, 1] = [0.10, 0.55, 0.15, 0.12, 0.00, 0.08]  # aggressive# Product transitionsP[1, 0] = [0.15, 0.20, 0.25, 0.20, 0.00, 0.20]P[1, 1] = [0.05, 0.15, 0.35, 0.30, 0.00, 0.15]# Wishlist transitions (new state!)P[2, 0] = [0.00, 0.15, 0.20, 0.40, 0.00, 0.25]P[2, 1] = [0.00, 0.05, 0.10, 0.65, 0.05, 0.15]# Cart transitionsP[3, 0] = [0.00, 0.05, 0.05, 0.15, 0.45, 0.30]P[3, 1] = [0.00, 0.00, 0.02, 0.10, 0.72, 0.16]# Absorbing statesP[4, :] = [0, 0, 0, 0, 1, 0]  # Purchase (absorbing)P[5, :] = [0, 0, 0, 0, 0, 1]  # Exit (absorbing)R = np.array([0, 0, 0, 0, 10, -1])  # rewards per stategamma = 0.9def value_iteration(P, R, gamma, policy, n_iter=200):    V = np.zeros(S)    for _ in range(n_iter):        V_new = np.copy(V)        for s in range(S-2):  # skip absorbing states            V_new[s] = R[s] + gamma * np.dot(P[s, policy[s]], V)        V = V_new    return Vno_rec_policy  = np.zeros(S, dtype=int)agg_rec_policy = np.ones(S, dtype=int)V_no  = value_iteration(P, R, gamma, no_rec_policy)V_agg = value_iteration(P, R, gamma, agg_rec_policy)print("State Values Comparison:")print(f"{'State':<12} {'No Rec V':>10} {'Agg Rec V':>10} {'Diff':>8}")print("-" * 44)for i, s in enumerate(states):    print(f"{s:<12} {V_no[i]:>10.3f} {V_agg[i]:>10.3f} {V_agg[i]-V_no[i]:>+8.3f}")print(f"Wishlist increases Homepage value by: {V_agg[0]-V_no[0]:+.3f}")# YOUR TURN: Try gamma=0.5 vs gamma=0.99 — does the Wishlist state# matter more or less with a longer planning horizon?

### Solution 3.2: Optimal Discount Factor Crossover

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Simple 5-state clickstream (without Wishlist)states_5 = ['Homepage','Product','Cart','Purchase','Exit']P5 = np.zeros((5, 2, 5))P5[0,0] = [0.3, 0.4, 0.1, 0.0, 0.2]P5[0,1] = [0.1, 0.6, 0.2, 0.0, 0.1]P5[1,0] = [0.2, 0.2, 0.3, 0.0, 0.3]P5[1,1] = [0.1, 0.2, 0.5, 0.0, 0.2]P5[2,0] = [0.0, 0.1, 0.2, 0.4, 0.3]P5[2,1] = [0.0, 0.0, 0.1, 0.7, 0.2]P5[3,:] = [0,0,0,1,0]; P5[4,:] = [0,0,0,0,1]R5 = np.array([0,0,0,10,-1])def vi5(P, R, gamma, policy, n_iter=500):    V = np.zeros(5)    for _ in range(n_iter):        Vn = np.copy(V)        for s in range(3):            Vn[s] = R[s] + gamma * np.dot(P[s, policy[s]], V)        V = Vn    return V[0]  # return Homepage valuegammas = np.linspace(0.1, 0.99, 100)V_no  = [vi5(P5, R5, g, np.zeros(5,dtype=int)) for g in gammas]V_agg = [vi5(P5, R5, g, np.ones(5,dtype=int)) for g in gammas]crossover_idx = np.argmax(np.array(V_agg) > np.array(V_no))print(f"Aggressive rec outperforms at γ = {gammas[crossover_idx]:.3f}")print("Intuition: low γ → agent values immediate reward → both policies similar")print("           high γ → future purchase matters more → aggressive rec wins")fig, ax = plt.subplots(figsize=(9,4))ax.plot(gammas, V_no,  label='No recommendation',  color='steelblue', linewidth=2)ax.plot(gammas, V_agg, label='Aggressive recommendation', color='seagreen', linewidth=2)ax.axvline(gammas[crossover_idx], color='red', linestyle='--',           label=f'Crossover γ≈{gammas[crossover_idx]:.2f}')ax.set_xlabel('Discount Factor γ')ax.set_ylabel('V(Homepage)')ax.set_title('When Does Aggressive Recommendation Become Worth It?')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()# YOUR TURN: Does the crossover γ depend on the reward magnitude?# Try R[Purchase] = 5, 10, 50 and plot how crossover shifts

### Solution 3.3: Non-Markov Process Demonstration

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)T = 500# AR(2) process: x_t = 0.6*x_{t-1} + 0.3*x_{t-2} + noisex = np.zeros(T)x[0] = np.random.randn()x[1] = np.random.randn()for t in range(2, T):    x[t] = 0.6 * x[t-1] + 0.3 * x[t-2] + np.random.normal(0, 0.3)# 1-state Markov model: predict x_t from x_{t-1} onlyfrom numpy.polynomial import polynomial as PX1 = x[1:-1].reshape(-1,1)  # x_{t-1}y  = x[2:]                   # x_t# Simple linear regression: x_t ≈ a * x_{t-1}a1 = np.dot(X1[:,0], y) / np.dot(X1[:,0], X1[:,0])resid_1state = y - a1 * X1[:,0]mse_1state = np.mean(resid_1state**2)# 2-state (augmented) model: predict x_t from (x_{t-1}, x_{t-2})X2 = np.column_stack([x[1:-1], x[:-2]])# Normal equations: coeffs = (X^T X)^{-1} X^T ycoeffs_2 = np.linalg.lstsq(X2, y, rcond=None)[0]resid_2state = y - X2 @ coeffs_2mse_2state = np.mean(resid_2state**2)print(f"1-state Markov model MSE: {mse_1state:.4f}")print(f"2-state augmented model MSE: {mse_2state:.4f}")print(f"Improvement: {(mse_1state-mse_2state)/mse_1state*100:.1f}%")print(f"2-state coefficients: a1={coeffs_2[0]:.3f} (true 0.6), a2={coeffs_2[1]:.3f} (true 0.3)")print("The augmented state space recovers the true AR(2) parameters!")# YOUR TURN: Plot the residuals from both models# Do the 1-state residuals show autocorrelation? (use plt.acorr)# Do the 2-state residuals look like white noise?